# Lab02 数学化讲解与函数实现（MLP / Accuracy / Temperature CE / XOR）

本笔记回答 5 个问题：
1. 什么是线性的前向传播函数
2. 什么是计算分类准确率的函数
3. 什么是两层多层感知机（MLP）
4. 什么是带温度缩放的交叉熵损失函数
5. XOR 问题是什么

并且每个“实现函数”都从数学角度说明：输入、输出、过程、使用到的机器学习函数。

## 0. 统一符号

- 批大小：$N$
- 输入维度：$d$
- 类别数：$C$
- 输入矩阵：$X \in \mathbb{R}^{N\times d}$
- 线性层参数：$W \in \mathbb{R}^{d\times C},\ b \in \mathbb{R}^{C}$
- logits：$Z \in \mathbb{R}^{N\times C}$（未归一化分数）
- 标签：$y \in \{0,1,\dots,C-1\}^N$

## 1) 线性的前向传播函数（Linear Forward）

### 数学定义
对一个 batch 的输入 $X$，线性前向传播是：
$$
Z = XW + b
$$
其中 $b$ 会广播到每个样本。

### 输入 / 输出 / 过程
- 输入：`x`，形状 `[N, d]`
- 输出：`logits`，形状 `[N, C]`
- 过程：矩阵乘法 + 偏置加法，不带非线性。

### 需要的函数（PyTorch）
- `nn.Linear(in_features=d, out_features=C)`：内部就是学 $W,b$ 并计算 $xW+b$。

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, in_dim] -> logits: [N, num_classes]
        return self.fc(x)

## 2) 计算分类准确率的函数（Accuracy）

### 数学定义
设第 $i$ 个样本的预测类别为
$$
\hat{y}_i = \arg\max_{c\in\{0,\dots,C-1\}} Z_{i,c}
$$
则准确率：
$$
\text{Acc} = \frac{1}{N}\sum_{i=1}^{N}\mathbf{1}(\hat{y}_i = y_i)
$$

### 输入 / 输出 / 过程
- 输入：`logits`（`[N, C]`），`y_true`（`[N]`）
- 输出：标量准确率（float）
- 过程：`argmax` 得到预测类别，再与真值逐元素比较并取均值。

### 需要的函数（PyTorch）
- `torch.argmax(logits, dim=1)`：取每行最大值对应的类别
- `(pred == y_true).float().mean()`：布尔值转 0/1 后求均值

In [ ]:
def accuracy_fn(logits: torch.Tensor, y_true: torch.Tensor) -> float:
    pred = torch.argmax(logits, dim=1)
    acc = (pred == y_true).float().mean()
    return acc.item()

## 3) 两层多层感知机（Two-layer MLP）

“两层 MLP”常指 **两个可学习的线性层**（中间有非线性激活）：
$$
h = \phi(XW_1 + b_1),\quad Z = hW_2 + b_2
$$
其中 $\phi$ 常用 ReLU：$\phi(t)=\max(0,t)$。

### 输入 / 输出 / 过程
- 输入：`x`（`[N, d]`）
- 输出：`logits`（`[N, C]`）
- 过程：
  1. 第一层线性变换到隐藏空间 `[N, H]`
  2. ReLU 引入非线性
  3. 第二层线性映射到类别 logits `[N, C]`

### 需要的函数（PyTorch）
- `nn.Linear(d, H)`
- `nn.ReLU()` 或 `F.relu`
- `nn.Linear(H, C)`

In [ ]:
class TwoLayerMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.act(self.fc1(x))
        logits = self.fc2(h)
        return logits

## 4) 带温度缩放的交叉熵损失（Temperature-scaled CE）

### 数学定义
先把 logits 用温度 $T>0$ 缩放：
$$
\tilde{Z} = \frac{Z}{T}
$$
然后做 softmax 概率：
$$
p_{i,c}=\frac{e^{\tilde{Z}_{i,c}}}{\sum_{k=1}^{C}e^{\tilde{Z}_{i,k}}}
$$
交叉熵：
$$
\mathcal{L}= -\frac{1}{N}\sum_{i=1}^{N}\log p_{i,y_i}
$$

### 温度的作用
- $T>1$：分布更平滑（模型更“保守”）
- $0<T<1$：分布更尖锐（模型更“自信”）

### 输入 / 输出 / 过程
- 输入：`logits [N,C]`，`y_true [N]`，`temperature`（正数）
- 输出：标量损失（`torch.Tensor`）
- 过程：先 `logits / T`，再调用交叉熵。

### 需要的函数（PyTorch）
- `F.cross_entropy(scaled_logits, y_true)`：内部已包含 `log_softmax + NLLLoss`，数值更稳定。

In [ ]:
def cross_entropy_with_temperature(
    logits: torch.Tensor,
    y_true: torch.Tensor,
    temperature: float = 1.0,
) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    scaled_logits = logits / temperature
    return F.cross_entropy(scaled_logits, y_true)

## 5) XOR 问题是什么

XOR（异或）二分类规则：两个二值输入不同则输出 1，相同则输出 0。

| $x_1$ | $x_2$ | $y=x_1\oplus x_2$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

### 数学本质
XOR **线性不可分**：不存在某个超平面 $w^Tx+b=0$ 能把正负类一次分开。

因此：
- 纯线性模型（单层线性分类器）无法拟合 XOR
- 含隐藏层 + 非线性的 MLP 可以拟合 XOR

这就是 MLP 相比线性模型的核心价值之一。

In [ ]:
# 一个最小 XOR 数据示例
x_xor = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y_xor = torch.tensor([0, 1, 1, 0])

print("x_xor shape:", x_xor.shape)
print("y_xor shape:", y_xor.shape)

## 6) 与你当前 lab02_utils.py 的对接方式

你的 `train_model(...)` 里调用方式是：
- `loss = loss_fn(y_pred, y_train, temperature)`（若有温度）
- `train_acc = accuracy_fn(y_pred_train, y_train)`

所以可以直接传：
- `loss_fn = cross_entropy_with_temperature`
- `accuracy_fn = accuracy_fn`
- `model = TwoLayerMLP(in_dim=2, hidden_dim=8, num_classes=2)`（例如 XOR）